# Content audit
> Runs as an **ArcGIS Online Hosted Notebook** (Standard runtime). `GIS('home')` uses the notebook's own session — no credentials in code. Logic lives in the `agolops` package (tested in CI); cells only orchestrate.

Flags stale, undocumented, unprotected, and orphaned items, and writes the findings to a CSV item for the content review meeting.

In [ ]:
# Install the tested logic (a hosted notebook starts from a clean runtime, without git)
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "https://github.com/iDan-Armstrong/agol-ops-notebooks/archive/refs/heads/main.zip",
])

In [ ]:
from arcgis.gis import GIS

from agolops.audit import audit_items, summarize

gis = GIS("home")
MAX_ITEMS = 2000

In [ ]:
# Pull the item properties the audit needs (plain dicts, so the logic stays testable)
KEYS = ["id", "title", "owner", "modified", "numViews", "description",
        "thumbnail", "access", "licenseInfo", "protected"]

items = [
    {k: getattr(item, k, None) for k in KEYS}
    for item in gis.content.search(query="", max_items=MAX_ITEMS)
]
org_usernames = {u.username for u in gis.users.search(max_users=1000)}
owners_gone = {i["owner"] for i in items} - org_usernames
print(f"{len(items)} items, {len(owners_gone)} departed owners")

In [ ]:
findings = audit_items(items, orphan_owners=owners_gone)
summarize(findings)

In [ ]:
import pandas as pd

df = pd.DataFrame([f.__dict__ for f in findings])
df.to_csv("/arcgis/home/content_audit.csv", index=False)
df.head(20)

The CSV lands in the notebook workspace (`/arcgis/home`), from where it can be added as a portal item or mailed to item owners. Schedule monthly via the notebook's **Tasks** pane.